# 03 - Camada Gold: tabelas analíticas

Lê a Silver (`datahackers_silver`) e grava no database `datahackers_gold` separando por perguntas

| Tabela | Pergunta do roteiro |
|---|---|
| `gold_estrutura_mercado` | Como está estruturado o mercado brasileiro de dados? |
| `gold_remuneracao_perfis` | Quais perfis profissionais são mais valorizados? |
| `gold_comparacoes_remuneracao` | Diferenças entre regiões, senioridades, modelos de trabalho, gênero e uso de IA |
| `gold_diversidade_genero` | Cenário de diversidade de gênero |
| `gold_adocao_tecnologias` | Tecnologias com maior adoção |
| `gold_adocao_ia` | Índice de adoção de IA |
| `gold_ia_empresas` | Uso, impacto e desafios da IA nas empresas (oportunidades e desafios) |

**Regras usadas em todas as tabelas**
- Percentuais calculados sobre as respostas válidas (sem nulos) de cada edição.
- Cruzamentos com menos de 30 respostas são descartados.
- "Ganha 12 mil reais ou mais" = limite inferior da faixa salarial a partir de 12.001 reais.
- Salário estimado = ponto médio da faixa; na faixa aberta (acima de 40.001 reais) usa-se o piso (estimativa conservadora).

**Como o notebook está organizado**
- Cada tabela Gold é uma consulta em **Spark SQL** (`spark.sql`), no mesmo estilo dos `CREATE TABLE AS SELECT` do Athena:
  um `WITH` com os passos intermediários e um `SELECT` final.
- O Python é usado só para gravar as tabelas no S3 e no catálogo (`salvar_gold`) e, nas perguntas de múltipla escolha,
  para montar a lista de colunas de opções de cada edição.

## 1. Configuração da sessão

In [5]:
%idle_timeout 30
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2


## 2. Contexto Spark e bibliotecas

In [1]:
import re

from pyspark.context import SparkContext
from awsglue.context import GlueContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

print("Sessão Spark pronta para trabalhar na Gold.")

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 30
Session ID: 0b9c3035-fcf7-4b43-b706-d72fa5f4e97c
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 0b9c3035-fcf7-4b43-b706-d72fa5f4e97c to get into ready status...
Session 0b9c3035-fcf7-4b43-b706-d72fa5f4e97c has been created.
Sessão Spark pronta para trabalhar na Gold.


## 3. Parâmetros do projeto

In [2]:
BUCKET = "teste-fiap-fase3"
DATABASE_SILVER = "datahackers_silver"
DATABASE_GOLD = "datahackers_gold"
DESTINO_GOLD = f"s3://{BUCKET}/gold"

# Edição -> nome da tabela Silver da edição e nome curto da view usada no SQL
EDICOES = {
    "2023": ("silver_state_of_data_2023", "silver_2023"),
    "2024": ("silver_state_of_data_2024", "silver_2024"),
    "2025-2026": ("silver_state_of_data_2025_2026", "silver_2025_2026"),
}

MIN_RESPOSTAS = 30            # tamanho mínimo de um grupo nos cruzamentos
LIMITE_SALARIO_ALTO = 12001   # início da faixa "R$ 12 mil ou mais"

## 4. Views da Silver

As views temporárias funcionam como "apelidos" para as tabelas da Silver dentro desta sessão.
- `perfil`: tabela unificada, com duas colunas de apoio (cargo de referência e status de uso de IA).
- `base_salarial`: só quem informou a faixa salarial, com o salário estimado e o indicador de 12 mil reais ou mais.
- `silver_2023`, `silver_2024`, `silver_2025_2026`: tabelas por edição, usadas nas perguntas de múltipla escolha.

In [3]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW perfil AS
SELECT *,
       COALESCE(cargo, cargo_gestor) AS cargo_referencia,     -- gestores respondem o cargo em outra pergunta
       CASE WHEN ia_usa_alguma THEN 'Usa IA'
            WHEN ia_nao_usa    THEN 'Não usa IA'
       END AS uso_ia_status
FROM {DATABASE_SILVER}.silver_perfil_profissionais
""")
spark.sql("CACHE TABLE perfil")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW base_salarial AS
SELECT *,
       CASE WHEN salario_max IS NULL THEN salario_min           -- faixa aberta: usa o piso
            ELSE (salario_min + salario_max) / 2
       END AS salario_estimado,
       CASE WHEN salario_min >= {LIMITE_SALARIO_ALTO} THEN 1 ELSE 0 END AS salario_12k_mais
FROM perfil
WHERE salario_min IS NOT NULL
""")

for ano, (tabela, view) in EDICOES.items():
    spark.sql(f"CREATE OR REPLACE TEMP VIEW {view} AS SELECT * FROM {DATABASE_SILVER}.{tabela}")

spark.sql("SELECT ano_pesquisa, COUNT(*) AS registros FROM perfil GROUP BY ano_pesquisa ORDER BY ano_pesquisa").show()
spark.sql("SELECT COUNT(*) AS respostas_com_faixa_salarial FROM base_salarial").show()

+------------+---------+
|ano_pesquisa|registros|
+------------+---------+
|        2023|     5293|
|        2024|     5215|
|   2025-2026|     3494|
+------------+---------+

+----------------------------+
|respostas_com_faixa_salarial|
+----------------------------+
|                       12841|
+----------------------------+


## 5. Funções de apoio

In [4]:
tabelas_gold = []

# Executa a consulta SQL, grava o resultado em Parquet Snappy e registra no catálogo da Gold.
def salvar_gold(nome, consulta):
    tabela = f"{DATABASE_GOLD}.gold_{nome}"
    caminho = f"{DESTINO_GOLD}/{nome}/"
    (
        spark.sql(consulta).write
        .mode("overwrite")
        .format("parquet")
        .option("compression", "snappy")
        .option("path", caminho)
        .saveAsTable(tabela)
    )
    tabelas_gold.append(f"gold_{nome}")
    total = spark.table(tabela).count()
    print(f"gold_{nome:<28} {total:>5} linhas | {caminho}")

# Mostra um resultado com uma coluna por edição, só para conferência visual.
def por_edicao(tabela, linha, valor, filtro):
    spark.sql(f"""
        SELECT * FROM (
            SELECT {linha}, ano_pesquisa, {valor}
            FROM {DATABASE_GOLD}.{tabela}
            WHERE {filtro}
        )
        PIVOT (FIRST({valor}) FOR ano_pesquisa IN ('2023', '2024', '2025-2026'))
        ORDER BY {linha}
    """).show(30, truncate=False)

## 6. Q1 - Estrutura do mercado

Cada `SELECT` do `UNION ALL` empilha um indicador. Depois contamos as respostas válidas
e calculamos o percentual dentro de cada edição e indicador (`SUM(...) OVER (PARTITION BY ...)`).

In [5]:
gold_estrutura_mercado = """
WITH respostas AS (
              SELECT ano_pesquisa, 'Situação de trabalho' AS indicador, CAST(situacao_trabalho AS STRING) AS categoria FROM perfil
    UNION ALL SELECT ano_pesquisa, 'Setor',              CAST(setor AS STRING)                    FROM perfil
    UNION ALL SELECT ano_pesquisa, 'Cargo',              CAST(cargo_referencia AS STRING)         FROM perfil
    UNION ALL SELECT ano_pesquisa, 'Senioridade',        CAST(senioridade_harmonizada AS STRING)  FROM perfil
    UNION ALL SELECT ano_pesquisa, 'Faixa etária',       CAST(faixa_idade AS STRING)              FROM perfil
    UNION ALL SELECT ano_pesquisa, 'Escolaridade',       CAST(escolaridade AS STRING)             FROM perfil
    UNION ALL SELECT ano_pesquisa, 'Região',             CAST(regiao AS STRING)                   FROM perfil
    UNION ALL SELECT ano_pesquisa, 'Modelo de trabalho', CAST(modelo_trabalho_resumido AS STRING) FROM perfil
),
contagem AS (
    SELECT ano_pesquisa, indicador, categoria, COUNT(*) AS quantidade
    FROM respostas
    WHERE categoria IS NOT NULL                                  -- só respostas válidas
    GROUP BY ano_pesquisa, indicador, categoria
)
SELECT ano_pesquisa, indicador, categoria, quantidade,
       SUM(quantidade) OVER (PARTITION BY ano_pesquisa, indicador) AS total_validas,
       ROUND(100 * quantidade / SUM(quantidade) OVER (PARTITION BY ano_pesquisa, indicador), 2) AS percentual
FROM contagem
"""
salvar_gold("estrutura_mercado", gold_estrutura_mercado)

for indicador in ["Situação de trabalho", "Senioridade", "Modelo de trabalho"]:
    print(f"\n=== {indicador} (% das respostas válidas) ===")
    por_edicao("gold_estrutura_mercado", "categoria", "percentual", f"indicador = '{indicador}'")

gold_estrutura_mercado              251 linhas | s3://teste-fiap-fase3/gold/estrutura_mercado/

=== Situação de trabalho (% das respostas válidas) ===
+---------------------------------------------------------------+-----+-----+---------+
|categoria                                                      |2023 |2024 |2025-2026|
+---------------------------------------------------------------+-----+-----+---------+
|Desempregado e não estou buscando recolocação                  |0.19 |0.21 |0.29     |
|Desempregado, buscando recolocação                             |6.82 |3.93 |4.78     |
|Empreendedor ou Empregado (CNPJ)                               |10.49|9.47 |12.25    |
|Empregado (CLT)                                                |68.09|72.52|65.2     |
|Estagiário                                                     |3.91 |3.57 |3.61     |
|Freelancer                                                     |0.96 |0.79 |1.06     |
|Prefiro não informar                                    

## 7. Q2 - Perfis mais valorizados (cargo × senioridade)

In [6]:
gold_remuneracao_perfis = f"""
WITH perfis AS (
    SELECT ano_pesquisa,
           cargo_referencia AS cargo,
           senioridade_harmonizada,
           COUNT(*)                               AS respostas,
           SUM(salario_12k_mais)                  AS respostas_12k_mais,
           ROUND(100 * AVG(salario_12k_mais), 2)  AS pct_12k_mais,
           ROUND(AVG(salario_estimado), 0)        AS salario_medio_estimado
    FROM base_salarial
    WHERE cargo_referencia IS NOT NULL
      AND senioridade_harmonizada IS NOT NULL
    GROUP BY ano_pesquisa, cargo_referencia, senioridade_harmonizada
    HAVING COUNT(*) >= {MIN_RESPOSTAS}
)
SELECT *,
       ROW_NUMBER() OVER (PARTITION BY ano_pesquisa ORDER BY pct_12k_mais DESC, respostas DESC) AS posicao
FROM perfis
"""
salvar_gold("remuneracao_perfis", gold_remuneracao_perfis)

print("\nTop 5 perfis por edição (% que ganha R$ 12 mil ou mais):")
spark.sql(f"""
SELECT ano_pesquisa, posicao, cargo, senioridade_harmonizada, respostas, pct_12k_mais, salario_medio_estimado
FROM {DATABASE_GOLD}.gold_remuneracao_perfis
WHERE posicao <= 5
ORDER BY ano_pesquisa, posicao
""").show(15, truncate=False)

gold_remuneracao_perfis              88 linhas | s3://teste-fiap-fase3/gold/remuneracao_perfis/

Top 5 perfis por edição (% que ganha R$ 12 mil ou mais):
+------------+-------+-------------------------------------------------------------------+-----------------------+---------+------------+----------------------+
|ano_pesquisa|posicao|cargo                                                              |senioridade_harmonizada|respostas|pct_12k_mais|salario_medio_estimado|
+------------+-------+-------------------------------------------------------------------+-----------------------+---------+------------+----------------------+
|2023        |1      |Gerente/Head                                                       |Gestor                 |318      |85.85       |21468.0               |
|2023        |2      |Diretor/VP                                                         |Gestor                 |39       |84.62       |28783.0               |
|2023        |3      |Cientista de Dados/

## 8. Q3, Q5 e Q6 - Comparações de remuneração

In [7]:
gold_comparacoes_remuneracao = f"""
WITH dimensoes AS (
              SELECT ano_pesquisa, 'Senioridade' AS dimensao, CAST(senioridade_harmonizada AS STRING) AS categoria, salario_12k_mais, salario_estimado FROM base_salarial
    UNION ALL SELECT ano_pesquisa, 'Região',             CAST(regiao AS STRING),                   salario_12k_mais, salario_estimado FROM base_salarial
    UNION ALL SELECT ano_pesquisa, 'Modelo de trabalho', CAST(modelo_trabalho_resumido AS STRING), salario_12k_mais, salario_estimado FROM base_salarial
    UNION ALL SELECT ano_pesquisa, 'Gênero',             CAST(genero AS STRING),                   salario_12k_mais, salario_estimado FROM base_salarial
    UNION ALL SELECT ano_pesquisa, 'Raça/etnia',         CAST(raca_etnia AS STRING),               salario_12k_mais, salario_estimado FROM base_salarial
    UNION ALL SELECT ano_pesquisa, 'Setor',              CAST(setor AS STRING),                    salario_12k_mais, salario_estimado FROM base_salarial
    UNION ALL SELECT ano_pesquisa, 'Uso de IA',          uso_ia_status,                            salario_12k_mais, salario_estimado FROM base_salarial
)
SELECT ano_pesquisa, dimensao, categoria,
       COUNT(*)                               AS respostas,
       SUM(salario_12k_mais)                  AS respostas_12k_mais,
       ROUND(100 * AVG(salario_12k_mais), 2)  AS pct_12k_mais,
       ROUND(AVG(salario_estimado), 0)        AS salario_medio_estimado
FROM dimensoes
WHERE categoria IS NOT NULL
GROUP BY ano_pesquisa, dimensao, categoria
HAVING COUNT(*) >= {MIN_RESPOSTAS}
"""
salvar_gold("comparacoes_remuneracao", gold_comparacoes_remuneracao)

for dimensao in ["Senioridade", "Gênero", "Região", "Modelo de trabalho", "Uso de IA"]:
    print(f"\n=== {dimensao}: % que ganha R$ 12 mil ou mais ===")
    por_edicao("gold_comparacoes_remuneracao", "categoria", "pct_12k_mais", f"dimensao = '{dimensao}'")

gold_comparacoes_remuneracao        122 linhas | s3://teste-fiap-fase3/gold/comparacoes_remuneracao/

=== Senioridade: % que ganha R$ 12 mil ou mais ===
+---------+-----+-----+---------+
|categoria|2023 |2024 |2025-2026|
+---------+-----+-----+---------+
|Gestor   |70.09|77.13|75.79    |
|Júnior   |0.76 |1.38 |1.54     |
|Pleno    |8.55 |9.3  |11.1     |
|Sênior+  |49.61|56.74|61.23    |
+---------+-----+-----+---------+


=== Gênero: % que ganha R$ 12 mil ou mais ===
+---------+-----+-----+---------+
|categoria|2023 |2024 |2025-2026|
+---------+-----+-----+---------+
|Feminino |23.58|28.31|36.95    |
|Masculino|32.91|40.79|44.66    |
+---------+-----+-----+---------+


=== Região: % que ganha R$ 12 mil ou mais ===
+------------+-----+-----+---------+
|categoria   |2023 |2024 |2025-2026|
+------------+-----+-----+---------+
|Centro-oeste|31.25|31.33|38.01    |
|Nordeste    |19.96|27.52|31.61    |
|Norte       |11.59|22.81|44.44    |
|Sudeste     |33.39|41.17|44.66    |
|Sul         |25

## 9. Q3 - Diversidade de gênero

In [8]:
gold_diversidade_genero = """
WITH recortes AS (
    SELECT ano_pesquisa, 'Geral' AS recorte_tipo, 'Geral' AS recorte, genero
    FROM perfil
    WHERE genero IS NOT NULL
    UNION ALL
    SELECT ano_pesquisa, 'Senioridade', senioridade_harmonizada, genero
    FROM perfil
    WHERE genero IS NOT NULL AND senioridade_harmonizada IS NOT NULL
),
contagem AS (
    SELECT ano_pesquisa, recorte_tipo, recorte, genero, COUNT(*) AS quantidade
    FROM recortes
    GROUP BY ano_pesquisa, recorte_tipo, recorte, genero
)
SELECT *,
       SUM(quantidade) OVER (PARTITION BY ano_pesquisa, recorte_tipo, recorte) AS total_recorte,
       ROUND(100 * quantidade / SUM(quantidade) OVER (PARTITION BY ano_pesquisa, recorte_tipo, recorte), 2) AS percentual
FROM contagem
"""
salvar_gold("diversidade_genero", gold_diversidade_genero)

print("\n% de mulheres no geral e por senioridade:")
por_edicao("gold_diversidade_genero", "recorte", "percentual", "genero = 'Feminino'")

gold_diversidade_genero              56 linhas | s3://teste-fiap-fase3/gold/diversidade_genero/

% de mulheres no geral e por senioridade:
+-------+-----+-----+---------+
|recorte|2023 |2024 |2025-2026|
+-------+-----+-----+---------+
|Geral  |24.43|23.49|21.95    |
|Gestor |18.75|19.04|18.43    |
|Júnior |28.59|27.88|28.19    |
|Pleno  |24.21|26.53|22.32    |
|Sênior+|21.63|21.31|20.55    |
+-------+-----+-----+---------+


## 10. Perguntas de múltipla escolha

Nas tabelas Silver por edição, cada pergunta de múltipla escolha tem uma coluna com o texto das respostas
(ex.: `p4_d_quais_das_linguagens...`) e uma coluna 0/1 por opção (ex.: `p4_d_3_python`).
O código e o texto das perguntas mudam entre as edições, por isso cada tema usa um de-para explícito
(coluna da pergunta em cada edição).

Como a lista de opções é diferente em cada edição (de 7 a 33 colunas), o Python monta essa lista e a consulta SQL
usa `stack()`, que transforma as N colunas de opção em N linhas `(opcao, marcou)`. Depois é só um `GROUP BY opcao`.
Perguntas de resposta única (sem colunas 0/1) são contadas pela distribuição das respostas.
A célula abaixo imprime, como exemplo, a consulta gerada para a pergunta de cloud de 2025-2026.

In [9]:
PADRAO_PERGUNTA = re.compile(r"^((?:p\d+|c_\d+)_[a-z])_")

# Colunas 0/1 de uma pergunta: mesmo prefixo da pergunta + número da opção (ex.: c_4_e_ -> c_4_e_1_aws...).
def opcoes_da_pergunta(colunas, coluna):
    prefixo = PADRAO_PERGUNTA.match(coluna).group(1) + "_"
    return prefixo, [c for c in colunas if re.match(re.escape(prefixo) + r"\d+_", c)]

# Rótulo da opção a partir do nome da coluna (ex.: c_4_e_1_amazon_web_services_aws -> Amazon web services aws).
def rotulo_opcao(coluna, prefixo):
    texto = re.sub("^" + re.escape(prefixo) + r"\d+_", "", coluna).replace("_", " ").strip()
    return texto[:1].upper() + texto[1:]

# Monta a consulta SQL que conta quantos respondentes marcaram cada opção de uma pergunta, em uma edição.
def sql_pergunta(pergunta, ano, coluna):
    view = EDICOES[ano][1]
    prefixo, opcoes = opcoes_da_pergunta(spark.table(view).columns, coluna)
    respondeu = f"{coluna} IS NOT NULL AND TRIM({coluna}) <> ''"

    if opcoes:
        pares = ",\n                   ".join(f"'{rotulo_opcao(o, prefixo)}', CAST({o} AS STRING)" for o in opcoes)
        print(f"{pergunta:<38} {ano:<10} {coluna[:60]:<60} | {len(opcoes)} opções")
        return f"""
    SELECT '{ano}' AS ano_pesquisa, '{pergunta}' AS pergunta, '{coluna}' AS coluna_origem,
           opcao,
           COUNT(*) AS respondentes,                -- uma linha por respondente em cada opção
           SUM(CASE WHEN LOWER(TRIM(marcou)) IN ('1', '1.0', 'true', 'sim') THEN 1 ELSE 0 END) AS mencoes
    FROM (
        SELECT stack({len(opcoes)},
                   {pares}) AS (opcao, marcou)
        FROM {view}
        WHERE {respondeu}
    )
    GROUP BY opcao"""

    # Resposta única: conta a distribuição das respostas.
    print(f"{pergunta:<38} {ano:<10} {coluna[:60]:<60} | resposta única")
    return f"""
    SELECT '{ano}' AS ano_pesquisa, '{pergunta}' AS pergunta, '{coluna}' AS coluna_origem,
           TRIM({coluna}) AS opcao,
           SUM(COUNT(*)) OVER () AS respondentes,   -- total de quem respondeu na edição
           COUNT(*) AS mencoes
    FROM {view}
    WHERE {respondeu}
    GROUP BY TRIM({coluna})"""

# Junta (UNION ALL) as consultas de várias perguntas e edições em uma view.
def criar_view_contagens(nome_view, perguntas):
    consultas = []
    for pergunta, colunas_por_ano in perguntas.items():
        for ano, coluna in colunas_por_ano.items():
            if coluna is None:
                print(f"{pergunta:<38} {ano:<10} sem pergunta equivalente nesta edição")
            else:
                consultas.append(sql_pergunta(pergunta, ano, coluna))
    spark.sql("\nUNION ALL".join(consultas)).createOrReplaceTempView(nome_view)

print(sql_pergunta("Cloud", "2025-2026", "c_4_e_cloud_dia_a_dia"))

Cloud                                  2025-2026  c_4_e_cloud_dia_a_dia                                        | 7 opções

    SELECT '2025-2026' AS ano_pesquisa, 'Cloud' AS pergunta, 'c_4_e_cloud_dia_a_dia' AS coluna_origem,
           opcao,
           COUNT(*) AS respondentes,                -- uma linha por respondente em cada opção
           SUM(CASE WHEN LOWER(TRIM(marcou)) IN ('1', '1.0', 'true', 'sim') THEN 1 ELSE 0 END) AS mencoes
    FROM (
        SELECT stack(7,
                   'Amazon web services aws', CAST(c_4_e_1_amazon_web_services_aws AS STRING),
                   'Google cloud gcp', CAST(c_4_e_2_google_cloud_gcp AS STRING),
                   'Azure microsoft', CAST(c_4_e_3_azure_microsoft AS STRING),
                   'Oracle cloud', CAST(c_4_e_4_oracle_cloud AS STRING),
                   'Ibm', CAST(c_4_e_5_ibm AS STRING),
                   'Servidores on premise nao utilizamos cloud', CAST(c_4_e_6_servidores_on_premise_nao_utilizamos_cloud AS STRING),
    

## 11. Q4 - Adoção de tecnologias

Observações de comparabilidade:
- **Linguagens (uso no dia a dia)**: a edição 2025-2026 não tem essa pergunta, apenas "linguagem preferida".
- **Cloud**: em 2023 o rótulo da pergunta fala em "cloud preferida", mas é a pergunta de múltipla escolha
  na mesma posição da "cloud no dia a dia" de 2024 (a preferida de resposta única é `p4_i`).
- **Fontes de dados**: em 2023 a pergunta é "já analisou ou processou"; em 2024 e 2025-2026, "no dia a dia".

In [10]:
TEMAS_TECNOLOGIA = {
    "Linguagens": {
        "2023": "p4_d_quais_das_linguagens_listadas_abaixo_voce_utiliza_no_trabalho",
        "2024": "c_4_d_linguagem_de_programacao_dia_a_dia",
        "2025-2026": None,
    },
    "Linguagem preferida": {
        "2023": "p4_f_entre_as_linguagens_listadas_abaixo_qual_e_a_sua_preferida",
        "2024": "c_4_f_linguagem_preferida",
        "2025-2026": "c_4_c_linguagem_preferida",
    },
    "Bancos de dados": {
        "2023": "p4_g_quais_dos_bancos_de_dados_fontes_de_dados_listados_abaixo_voce_utiliza_no_trabalho",
        "2024": "c_4_g_banco_de_dados_dia_a_dia",
        "2025-2026": "c_4_d_banco_de_dados_dia_a_dia",
    },
    "Cloud": {
        "2023": "p4_h_dentre_as_opcoes_listadas_qual_sua_cloud_preferida",
        "2024": "c_4_h_cloud_dia_a_dia",
        "2025-2026": "c_4_e_cloud_dia_a_dia",
    },
    "Ferramentas de BI": {
        "2023": "p4_j_ferramenta_de_bi_utilizada_no_dia_a_dia",
        "2024": "c_4_j_ferramenta_de_bi_dia_a_dia",
        "2025-2026": "c_4_g_ferramenta_de_bi_dia_a_dia",
    },
    "Fontes de dados": {
        "2023": "p4_b_quais_das_fontes_de_dados_listadas_voce_ja_analisou_ou_processou_no_trabalho",
        "2024": "c_4_b_fontes_de_dados_dia_a_dia",
        "2025-2026": "c_4_b_fontes_de_dados_dia_a_dia",
    },
}
criar_view_contagens("contagem_tecnologias", TEMAS_TECNOLOGIA)

gold_adocao_tecnologias = """
SELECT ano_pesquisa,
       pergunta AS tema,
       coluna_origem,
       opcao AS tecnologia,
       respondentes,
       mencoes,
       ROUND(100 * mencoes / respondentes, 2) AS percentual,
       ROW_NUMBER() OVER (PARTITION BY ano_pesquisa, pergunta ORDER BY mencoes DESC, opcao) AS posicao
FROM contagem_tecnologias
"""
salvar_gold("adocao_tecnologias", gold_adocao_tecnologias)

for tema in TEMAS_TECNOLOGIA:
    print(f"\n=== {tema}: top 8 por edição ===")
    spark.sql(f"""
        SELECT ano_pesquisa, posicao, tecnologia, percentual
        FROM {DATABASE_GOLD}.gold_adocao_tecnologias
        WHERE tema = '{tema}' AND posicao <= 8
        ORDER BY ano_pesquisa, posicao
    """).show(30, truncate=False)

Linguagens                             2023       p4_d_quais_das_linguagens_listadas_abaixo_voce_utiliza_no_tr | 15 opções
Linguagens                             2024       c_4_d_linguagem_de_programacao_dia_a_dia                     | 15 opções
Linguagens                             2025-2026  sem pergunta equivalente nesta edição
Linguagem preferida                    2023       p4_f_entre_as_linguagens_listadas_abaixo_qual_e_a_sua_prefer | resposta única
Linguagem preferida                    2024       c_4_f_linguagem_preferida                                    | resposta única
Linguagem preferida                    2025-2026  c_4_c_linguagem_preferida                                    | 10 opções
Bancos de dados                        2023       p4_g_quais_dos_bancos_de_dados_fontes_de_dados_listados_abai | 33 opções
Bancos de dados                        2024       c_4_g_banco_de_dados_dia_a_dia                               | 33 opções
Bancos de dados                        20

## 12. Q5 - Adoção de IA pelos profissionais

Os indicadores de IA da Silver são colunas verdadeiro/falso. A média de `CAST(... AS INT)` é a proporção de quem marcou.

In [11]:
gold_adocao_ia = f"""
WITH base AS (
    SELECT * FROM perfil WHERE uso_ia_trabalho IS NOT NULL       -- quem respondeu a pergunta de IA
),
recortes AS (
              SELECT 'Geral' AS recorte_tipo, 'Geral' AS recorte,             * FROM base
    UNION ALL SELECT 'Senioridade',        CAST(senioridade_harmonizada AS STRING),  * FROM base WHERE senioridade_harmonizada IS NOT NULL
    UNION ALL SELECT 'Gênero',             CAST(genero AS STRING),                   * FROM base WHERE genero IS NOT NULL
    UNION ALL SELECT 'Modelo de trabalho', CAST(modelo_trabalho_resumido AS STRING), * FROM base WHERE modelo_trabalho_resumido IS NOT NULL
),
agregado AS (
    SELECT ano_pesquisa, recorte_tipo, recorte,
           COUNT(*) AS respondentes,
           ROUND(100 * AVG(CAST(ia_usa_alguma   AS INT)), 2) AS usa_alguma,
           ROUND(100 * AVG(CAST(ia_nao_usa      AS INT)), 2) AS nao_usa,
           ROUND(100 * AVG(CAST(ia_gratuita     AS INT)), 2) AS gratuita,
           ROUND(100 * AVG(CAST(ia_paga_empresa AS INT)), 2) AS paga_empresa,
           ROUND(100 * AVG(CAST(ia_paga_propria AS INT)), 2) AS paga_propria,
           ROUND(100 * AVG(CAST(ia_codigo       AS INT)), 2) AS codigo
    FROM recortes
    GROUP BY ano_pesquisa, recorte_tipo, recorte
    HAVING COUNT(*) >= {MIN_RESPOSTAS}
)
          SELECT ano_pesquisa, recorte_tipo, recorte, respondentes, 'Usa alguma IA' AS indicador, usa_alguma AS percentual FROM agregado
UNION ALL SELECT ano_pesquisa, recorte_tipo, recorte, respondentes, 'Não usa IA',            nao_usa      FROM agregado
UNION ALL SELECT ano_pesquisa, recorte_tipo, recorte, respondentes, 'Ferramentas gratuitas', gratuita     FROM agregado
UNION ALL SELECT ano_pesquisa, recorte_tipo, recorte, respondentes, 'Empresa paga',          paga_empresa FROM agregado
UNION ALL SELECT ano_pesquisa, recorte_tipo, recorte, respondentes, 'Paga do próprio bolso', paga_propria FROM agregado
UNION ALL SELECT ano_pesquisa, recorte_tipo, recorte, respondentes, 'IA para código',        codigo       FROM agregado
"""
salvar_gold("adocao_ia", gold_adocao_ia)

print("\nAdoção de IA - geral (% de quem respondeu):")
por_edicao("gold_adocao_ia", "indicador", "percentual", "recorte_tipo = 'Geral'")

print("\nUsa alguma IA - por senioridade:")
por_edicao("gold_adocao_ia", "recorte", "percentual", "recorte_tipo = 'Senioridade' AND indicador = 'Usa alguma IA'")

gold_adocao_ia                      180 linhas | s3://teste-fiap-fase3/gold/adocao_ia/

Adoção de IA - geral (% de quem respondeu):
+---------------------+-----+-----+---------+
|indicador            |2023 |2024 |2025-2026|
+---------------------+-----+-----+---------+
|Empresa paga         |6.36 |19.27|42.33    |
|Ferramentas gratuitas|63.65|53.72|30.55    |
|IA para código       |11.77|23.72|29.83    |
|Não usa IA           |19.03|6.17 |1.85     |
|Paga do próprio bolso|6.92 |15.54|26.94    |
|Usa alguma IA        |80.97|93.83|98.15    |
+---------------------+-----+-----+---------+


Usa alguma IA - por senioridade:
+-------+-----+-----+---------+
|recorte|2023 |2024 |2025-2026|
+-------+-----+-----+---------+
|Júnior |80.78|94.71|97.18    |
|Pleno  |82.05|93.33|98.2     |
|Sênior+|80.04|93.83|98.48    |
+-------+-----+-----+---------+


## 13. Q5 e Q7 - IA nas empresas: uso, desafios e resultados

As perguntas da seção de gestores (P3) mostram a visão de quem lidera times de dados; a pergunta da seção
de profissionais (P4) mostra a visão de quem executa. Resultados com IA existem apenas em 2025-2026
e é uma pergunta de resposta única, lida da tabela unificada.

In [12]:
PERGUNTAS_IA_EMPRESA = {
    "Uso de IA na empresa (gestores)": {
        "2023": "p3_f_tipos_de_uso_de_ai_generativa_e_llms_na_empresa",
        "2024": "c_3_f_tipo_de_uso_de_ai_generativa_e_llm_na_empresa",
        "2025-2026": "c_3_f_tipo_de_uso_de_ai_generativa_e_llm_na_empresa",
    },
    "Motivos para não usar IA (gestores)": {
        "2023": "p3_g_motivos_que_levam_a_empresa_a_nao_usar_ai_genrativa_e_llms",
        "2024": "c_3_g_motivos_para_nao_usar_ai_generativa_e_llm",
        "2025-2026": "c_3_h_motivos_para_nao_usar_ai_generativa_e_llm",
    },
    "Uso de IA na empresa (profissionais)": {
        "2023": "p4_l_qual_o_tipo_de_uso_de_ai_generativa_e_llms_na_empresa",
        "2024": "c_4_l_tipo_de_uso_de_ai_generativa_e_llm_na_empresa",
        "2025-2026": "c_4_i_tipo_de_uso_de_ai_generativa_e_llm_na_empresa",
    },
}
criar_view_contagens("contagem_ia_empresas", PERGUNTAS_IA_EMPRESA)

gold_ia_empresas = """
WITH resultados AS (
    SELECT ano_pesquisa,
           'Resultados com IA na empresa (gestores)' AS pergunta,
           'c_3_g_empresa_esta_conseguindo_ter_bons_resultados_com_llms' AS coluna_origem,
           CAST(resultados_ia_empresa AS STRING) AS opcao,
           SUM(COUNT(*)) OVER (PARTITION BY ano_pesquisa) AS respondentes,
           COUNT(*) AS mencoes
    FROM perfil
    WHERE resultados_ia_empresa IS NOT NULL
    GROUP BY ano_pesquisa, resultados_ia_empresa
),
todas AS (
    SELECT * FROM contagem_ia_empresas
    UNION ALL
    SELECT * FROM resultados
)
SELECT ano_pesquisa, pergunta, coluna_origem, opcao, respondentes, mencoes,
       ROUND(100 * mencoes / respondentes, 2) AS percentual,
       ROW_NUMBER() OVER (PARTITION BY ano_pesquisa, pergunta ORDER BY mencoes DESC, opcao) AS posicao
FROM todas
"""
salvar_gold("ia_empresas", gold_ia_empresas)

for pergunta in list(PERGUNTAS_IA_EMPRESA) + ["Resultados com IA na empresa (gestores)"]:
    print(f"\n=== {pergunta}: top 5 por edição ===")
    spark.sql(f"""
        SELECT ano_pesquisa, posicao, opcao, percentual
        FROM {DATABASE_GOLD}.gold_ia_empresas
        WHERE pergunta = '{pergunta}' AND posicao <= 5
        ORDER BY ano_pesquisa, posicao
    """).show(20, truncate=70)

Uso de IA na empresa (gestores)        2023       p3_f_tipos_de_uso_de_ai_generativa_e_llms_na_empresa         | 8 opções
Uso de IA na empresa (gestores)        2024       c_3_f_tipo_de_uso_de_ai_generativa_e_llm_na_empresa          | 8 opções
Uso de IA na empresa (gestores)        2025-2026  c_3_f_tipo_de_uso_de_ai_generativa_e_llm_na_empresa          | 8 opções
Motivos para não usar IA (gestores)    2023       p3_g_motivos_que_levam_a_empresa_a_nao_usar_ai_genrativa_e_l | 9 opções
Motivos para não usar IA (gestores)    2024       c_3_g_motivos_para_nao_usar_ai_generativa_e_llm              | 9 opções
Motivos para não usar IA (gestores)    2025-2026  c_3_h_motivos_para_nao_usar_ai_generativa_e_llm              | 9 opções
Uso de IA na empresa (profissionais)   2023       p4_l_qual_o_tipo_de_uso_de_ai_generativa_e_llms_na_empresa   | 8 opções
Uso de IA na empresa (profissionais)   2024       c_4_l_tipo_de_uso_de_ai_generativa_e_llm_na_empresa          | 8 opções
Uso de IA na empresa (pr

## 14. Validação final pelo catálogo

In [13]:
# Releio cada tabela pelo catálogo e confiro se os percentuais estão entre 0 e 100.
consultas = []
for tabela in tabelas_gold:
    coluna = "pct_12k_mais" if "pct_12k_mais" in spark.table(f"{DATABASE_GOLD}.{tabela}").columns else "percentual"
    consultas.append(f"""
        SELECT '{tabela}' AS tabela,
               COUNT(*) AS linhas,
               COUNT(DISTINCT ano_pesquisa) AS edicoes,
               MIN({coluna}) AS menor_percentual,
               MAX({coluna}) AS maior_percentual,
               CASE WHEN MIN({coluna}) >= 0 AND MAX({coluna}) <= 100 THEN 'OK' ELSE 'FORA DO INTERVALO' END AS checagem
        FROM {DATABASE_GOLD}.{tabela}""")
spark.sql("\nUNION ALL".join(consultas)).show(truncate=False)

# Nas distribuições de resposta única, a soma dos percentuais por edição deve ser 100%.
spark.sql(f"""
SELECT COUNT(*) AS distribuicoes_com_soma_diferente_de_100
FROM (
    SELECT ano_pesquisa, indicador, ROUND(SUM(percentual), 1) AS soma
    FROM {DATABASE_GOLD}.gold_estrutura_mercado
    GROUP BY ano_pesquisa, indicador
)
WHERE soma < 99.5 OR soma > 100.5
""").show()

spark.sql(f"SHOW TABLES IN {DATABASE_GOLD}").show(truncate=False)

+----------------------------+------+-------+----------------+----------------+--------+
|tabela                      |linhas|edicoes|menor_percentual|maior_percentual|checagem|
+----------------------------+------+-------+----------------+----------------+--------+
|gold_estrutura_mercado      |251   |3      |0.11            |72.52           |OK      |
|gold_remuneracao_perfis     |88    |3      |0.0             |94.59           |OK      |
|gold_comparacoes_remuneracao|122   |3      |0.76            |77.13           |OK      |
|gold_diversidade_genero     |56    |3      |0.1             |81.57           |OK      |
|gold_adocao_tecnologias     |292   |3      |0.0             |93.35           |OK      |
|gold_adocao_ia              |180   |3      |1.08            |98.92           |OK      |
|gold_ia_empresas            |80    |3      |3.23            |50.65           |OK      |
+----------------------------+------+-------+----------------+----------------+--------+

+-------------------